# 🔗 Sequential and Conditional Workflows in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **Workflow vs agent** - Who decides the next step: the graph you wrote, or a model at run time
2. **Sequential workflow** - Chain prebuilt agents with `add_edge` so every input takes the same fixed path
3. **Conditional workflow** - Let a classifier agent fill a typed field, then branch on it with `add_conditional_edges`
4. **Mixing both** - A fixed sequence with one branch decided by plain code
5. **Who decides what** - Separate the steps the workflow fixes from the decisions an agent makes

## Where this fits
This is pattern 2 of [Five Practical Agent Patterns](../../../../docs/source/08/five-practical-agent-patterns.md) and the first notebook of this module, because it is the simplest: you know the stages before the run starts. Every agent here is built with LangChain's prebuilt `create_agent`; no agent loop is written by hand. LangGraph's `StateGraph` decides the order in which those agents run.

**Flow:** `Input → stage A → stage B → condition → stage C or D → result`

---
## 📦 Step 1: Environment Setup

The setup cell imports everything the notebook uses, loads the API keys from the project-root `.env` and sets `RECURSION_LIMIT`, the step cap every agent run passes.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and imports
# ============================================================================
import warnings
from typing import Literal, TypedDict

from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Every agent run passes this cap on LangGraph super-steps
RECURSION_LIMIT = 25

print(f"✅ Environment ready (recursion_limit = {RECURSION_LIMIT})")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: one chat model shared by every agent
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

In [ ]:
# ============================================================================
# ASK: run a prebuilt agent on one message and return its final reply
# ============================================================================
def ask(agent, text: str) -> str:
    """Send one user message to a create_agent agent; return the text of its last message."""
    result = agent.invoke({"messages": [HumanMessage(content=text)]}, {"recursion_limit": RECURSION_LIMIT})
    return result["messages"][-1].content


print("✅ ask() ready")

In [ ]:
# ============================================================================
# RUN_AND_TRACE: stream a workflow and record which nodes ran, in order
# ============================================================================
def run_and_trace(app, inputs: dict):
    """Run a compiled graph; return (node names in the order they ran, final state)."""
    order, state = [], dict(inputs)
    # stream_mode="updates" yields {node_name: what_that_node_wrote} after each node finishes
    for update in app.stream(inputs, stream_mode="updates"):
        for node, values in update.items():
            order.append(node)
            state.update(values or {})
    return order, state


print("✅ run_and_trace() ready")

---
## 🧭 Part 1: Workflow vs Agent

Both words describe a system that calls a model several times. They differ in **who chooses the next step**.

| | Workflow (this notebook) | Agent |
|---|---|---|
| Who picks the next step | the graph you wrote: edges and branch functions | the model, at run time, through tool calls |
| Set of possible paths | fixed before the run | open: any tool, in any order, any number of times |
| Predictability | high: the same input type takes the same path | lower: the path can differ run to run |
| Best when | the stages are known in advance | the steps cannot be listed in advance |

The two combine well. In this notebook each **stage** is an agent built with `create_agent` (it decides *what to write*), while the **workflow** decides *which stage runs next*.

- A **sequential** workflow has one path: every stage runs, in the same order, for every input.
- A **conditional** workflow adds at least one fork: after a stage, a check reads the state and picks one of several branches you wired in advance.

---
## ➡️ Part 2: Sequential Workflow - Research → Draft → Review

Three prebuilt agents, each with one job and no tools:

| Agent | Reads | Writes |
|---|---|---|
| `researcher` | the topic | short research notes |
| `writer` | the notes | one beginner-friendly paragraph |
| `editor` | the paragraph | the edited final text |

The agents never talk to each other. Each graph node calls one agent and writes its reply into a **typed state field** (`notes`, `draft`, `final`). The next node reads only the field it needs. This is the whole contract between stages.

In [ ]:
# ============================================================================
# SEQUENTIAL AGENTS: three prebuilt agents, one job each
# ============================================================================
researcher = create_agent(
    llm,
    system_prompt="You are a researcher. List 4-5 key facts about the topic as short bullet points. No introduction.",
    name="researcher",
)
writer = create_agent(
    llm,
    system_prompt=(
        "You are a writer. Turn the research notes you receive into one clear paragraph "
        "of at most 120 words for a beginner audience. Return only the paragraph."
    ),
    name="writer",
)
editor = create_agent(
    llm,
    system_prompt=(
        "You are an editor. Fix grammar and clarity in the paragraph you receive. "
        "Keep its facts and length. Return only the edited paragraph."
    ),
    name="editor",
)

print("✅ Agents ready: researcher, writer, editor")

In [ ]:
# ============================================================================
# SEQUENTIAL GRAPH: START -> research -> draft -> review -> END
# ============================================================================
class ArticleState(TypedDict):
    topic: str
    notes: str  # written by research
    draft: str  # written by draft
    final: str  # written by review


def research(state: ArticleState) -> dict:
    return {"notes": ask(researcher, f"Topic: {state['topic']}")}


def draft(state: ArticleState) -> dict:
    return {"draft": ask(writer, f"Research notes:\n{state['notes']}")}


def review(state: ArticleState) -> dict:
    return {"final": ask(editor, state["draft"])}


builder = StateGraph(ArticleState)
builder.add_node("research", research)
builder.add_node("draft", draft)
builder.add_node("review", review)
builder.add_edge(START, "research")
builder.add_edge("research", "draft")  # plain edges: no choice, ever
builder.add_edge("draft", "review")
builder.add_edge("review", END)
sequential_app = builder.compile()

print(sequential_app.get_graph().draw_mermaid())

The Mermaid text above is the whole workflow: one straight line of plain edges. Paste it into [mermaid.live](https://mermaid.live) to see it drawn.

> 💡 **Shortcut:** `builder.add_sequence([research, draft, review])` adds the three nodes and the edges between them in one call. The explicit edges are used here so the next part can change one of them into a branch.

### 2.1 ▶️ Two Topics, One Path

Two different topics go through the graph. A sequential workflow must take the same path both times; a ✅ check compares the recorded node order with the expected one.

In [ ]:
# ============================================================================
# RUN: TWO TOPICS THROUGH THE SEQUENTIAL WORKFLOW
# ============================================================================
EXPECTED_PATH = ["research", "draft", "review"]

for topic in ["How vaccines train the immune system", "Why the sky is blue"]:
    path, result = run_and_trace(sequential_app, {"topic": topic})
    print("=" * 80)
    print(f"TOPIC: {topic}")
    print(f"path:  {' → '.join(path)}")
    print(f"\n📝 notes:\n{result['notes']}")
    print(f"\n✍️ final:\n{result['final']}")
    print(f"\n{'✅' if path == EXPECTED_PATH else '❌'} path is the fixed sequence")

Both runs print `research → draft → review`. The *content* changed with the topic; the *path* did not. The input cannot change the path, because no edge offers a choice.

---
## 🔀 Part 3: Conditional Workflow - Route a Support Ticket

Now the path depends on the input. A support desk has three specialists, and each ticket should reach exactly one of them:

```
START → classify → billing   → END
                 → technical → END
                 → general   → END
```

The fork needs a decision. The decision is split into two parts on purpose:

1. **The classifier agent decides the category.** `create_agent(..., response_format=ToolStrategy(TicketRoute))` makes the agent return a validated `TicketRoute` object in `result["structured_response"]`. Its `category` must be one of three `Literal` values, so no free-text answer can reach the router.
2. **The workflow maps the category to a node.** `route_ticket` is plain code: it reads `state["category"]` and returns the branch name. `add_conditional_edges` lists every allowed branch in advance.

So the agent can pick only one of the three branches you wired. It cannot invent a fourth.

In [ ]:
# ============================================================================
# CLASSIFIER AGENT: typed output with ToolStrategy
# ============================================================================
class TicketRoute(BaseModel):
    """Where a support ticket should go."""

    category: Literal["billing", "technical", "general"] = Field(
        description=(
            "billing: payments, charges, refunds, invoices; "
            "technical: errors, crashes, bugs, login problems; "
            "general: everything else"
        )
    )
    reason: str = Field(description="One short sentence explaining the choice")


classifier = create_agent(
    llm,
    system_prompt="You classify customer support tickets. Pick exactly one category.",
    response_format=ToolStrategy(TicketRoute),  # validated by Pydantic; retried on a bad reply
    name="classifier",
)

SPECIALIST_PROMPTS = {
    "billing": "You are a billing specialist. Answer the customer's billing question in at most 3 sentences.",
    "technical": "You are a technical support engineer. Give at most 3 short troubleshooting steps.",
    "general": "You are a friendly customer service agent. Answer in at most 3 sentences.",
}
specialists = {
    name: create_agent(llm, system_prompt=prompt, name=f"{name}_agent")
    for name, prompt in SPECIALIST_PROMPTS.items()
}

print(f"✅ Classifier ready; specialists: {list(specialists)}")

In [ ]:
# ============================================================================
# CONDITIONAL GRAPH: classify, then branch with add_conditional_edges
# ============================================================================
class TicketState(TypedDict):
    ticket: str
    category: str  # written by classify; the branch condition reads it
    reason: str
    reply: str  # written by whichever specialist ran


def classify(state: TicketState) -> dict:
    result = classifier.invoke(
        {"messages": [HumanMessage(content=state["ticket"])]}, {"recursion_limit": RECURSION_LIMIT}
    )
    route = result["structured_response"]  # a TicketRoute instance
    return {"category": route.category, "reason": route.reason}


def make_specialist_node(agent):
    def specialist(state: TicketState) -> dict:
        return {"reply": ask(agent, state["ticket"])}

    return specialist


def route_ticket(state: TicketState) -> Literal["billing", "technical", "general"]:
    """The branch condition: plain code reading the classifier's typed output."""
    return state["category"]


builder = StateGraph(TicketState)
builder.add_node("classify", classify)
for name, agent in specialists.items():
    builder.add_node(name, make_specialist_node(agent))
    builder.add_edge(name, END)
builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route_ticket,
    {"billing": "billing", "technical": "technical", "general": "general"},  # every allowed branch
)
conditional_app = builder.compile()

print(conditional_app.get_graph().draw_mermaid())

In the Mermaid text, the edges out of `classify` are dotted (`-.->`): they are conditional. Exactly one of them fires per run.

### 3.1 ▶️ Three Tickets, Three Paths

In [ ]:
# ============================================================================
# RUN: THREE TICKETS THROUGH THE CONDITIONAL WORKFLOW
# ============================================================================
TICKETS = [  # (ticket, expected branch)
    ("I was charged twice for my subscription this month.", "billing"),
    ("The app crashes every time I open the settings page.", "technical"),
    ("Do you have an office in Bangalore?", "general"),
]

passed = 0
for ticket, expected in TICKETS:
    path, result = run_and_trace(conditional_app, {"ticket": ticket})
    ok = path == ["classify", expected]
    passed += ok
    print("=" * 80)
    print(f"TICKET:   {ticket}")
    print(f"path:     {' → '.join(path)}")
    print(f"category: {result['category']} ({result['reason']})")
    print(f"reply:    {result['reply']}")
    print(f"{'✅' if ok else '❌'} expected classify → {expected}")

print(f"\n{passed}/{len(TICKETS)} tickets took the expected branch")

Each ticket ran **two** nodes: `classify` plus one specialist. The other two specialists never ran, so their model calls cost nothing. That is the practical gain of a conditional workflow: only the needed branch runs.

---
## 🧩 Part 4: Mixing Both - A Sequence With One Code Branch

Real workflows are usually a sequence with a few forks. The pattern's flow line, `A → B → condition → C or D`, is exactly that.

Here the article pipeline from Part 2 gets a **length gate** after `draft`:
- the draft is within `word_limit` → go straight to `review`
- the draft is too long → `shorten` first, then `review`

This branch condition is **plain code** (a word count), not an agent. When a rule can be checked in code, check it in code: it is free, fast and always gives the same answer.

`build_gated_pipeline(word_limit)` returns a compiled graph, so the same workflow can run with two limits.

In [ ]:
# ============================================================================
# GATED PIPELINE: research -> draft -> (shorten?) -> review
# ============================================================================
shortener = create_agent(
    llm,
    system_prompt="Shorten the paragraph you receive to the word limit given. Keep the key facts. Return only the paragraph.",
    name="shortener",
)


def word_count(text: str) -> int:
    return len(text.split())


def build_gated_pipeline(word_limit: int):
    """Compile research -> draft -> length gate -> review, with the gate's limit fixed at build time."""

    def shorten(state: ArticleState) -> dict:
        return {"draft": ask(shortener, f"Word limit: {word_limit}\n\n{state['draft']}")}

    def length_gate(state: ArticleState) -> Literal["shorten", "review"]:
        return "shorten" if word_count(state["draft"]) > word_limit else "review"

    builder = StateGraph(ArticleState)
    builder.add_node("research", research)
    builder.add_node("draft", draft)
    builder.add_node("shorten", shorten)
    builder.add_node("review", review)
    builder.add_edge(START, "research")
    builder.add_edge("research", "draft")
    builder.add_conditional_edges("draft", length_gate, ["shorten", "review"])
    builder.add_edge("shorten", "review")
    builder.add_edge("review", END)
    return builder.compile()


print(build_gated_pipeline(60).get_graph().draw_mermaid())

### 4.1 ▶️ Same Topic, Two Limits

The writer aims for at most 120 words, so a limit of 150 should let the draft pass, and a limit of 40 should send it to `shorten`. The model decides how long the draft is, so the ✅ check does not hard-code a path. It recomputes the gate from the draft and confirms the workflow followed its own rule.

In [ ]:
# ============================================================================
# RUN: ONE TOPIC, TWO WORD LIMITS
# ============================================================================
TOPIC = "How solar panels turn sunlight into electricity"

for limit in [150, 40]:
    app = build_gated_pipeline(limit)
    # Stream manually here: we need the draft as written by `draft`, before `shorten` replaces it
    path, first_draft, result = [], None, {"topic": TOPIC}
    for update in app.stream({"topic": TOPIC}, stream_mode="updates"):
        for node, values in update.items():
            path.append(node)
            result.update(values or {})
            if node == "draft":
                first_draft = values["draft"]

    expected = ["research", "draft"] + (["shorten"] if word_count(first_draft) > limit else []) + ["review"]
    print("=" * 80)
    print(f"word_limit = {limit}; first draft = {word_count(first_draft)} words")
    print(f"path: {' → '.join(path)}")
    print(f"final ({word_count(result['final'])} words):\n{result['final']}")
    print(f"{'✅' if path == expected else '❌'} path follows the length gate")

---
## 🧑‍⚖️ Part 5: Who Decides What

The [study guide](../../../../docs/source/08/five-practical-agent-patterns.md) asks you to separate the steps the workflow fixes from the decisions an agent makes. For the three graphs above:

| Decision | Made by | Where |
|---|---|---|
| Order `research → draft → review` | workflow | `add_edge` calls |
| What the notes, draft and final text say | agents | `researcher`, `writer`, `editor` |
| Ticket category | agent | `classifier`, constrained to three `Literal` values |
| Which node handles each category | workflow | `route_ticket` and the `add_conditional_edges` mapping |
| Whether a draft needs shortening | workflow (code) | `length_gate`, a word count |
| When the run ends | workflow | the edges into `END` |

Agents fill in **content and typed decisions**; the graph turns those decisions into **control flow**. That split is what makes a workflow predictable and easy to test.

---
## 📝 Summary

In this notebook, we learned:

### 1. Workflow vs agent
- In a workflow, the graph picks the next step; in an agent, the model does. Here every stage is a prebuilt `create_agent` agent, and LangGraph orders the stages.

### 2. Sequential workflow
- Plain `add_edge` calls give one fixed path. Different inputs change the content, never the path.
- Stages communicate through typed state fields (`notes`, `draft`, `final`), not through each other's messages.

### 3. Conditional workflow
- A classifier agent with `response_format=ToolStrategy(TicketRoute)` returns a validated typed choice in `structured_response`.
- `add_conditional_edges` maps that choice to one of the branches listed in advance; only that branch runs.

### 4. Mixing both
- A branch condition can be plain code (`length_gate`). Use code whenever a rule can be checked without a model.

### 5. Who decides what
- Agents decide content and typed choices; the workflow decides the order, the branches and the end.

### Next Steps
- **`02_supervisor_agents_as_tools.ipynb`**: hand the routing decision itself to a supervisor agent, which picks specialists at run time